# 02 반도체 공정 데이터 분석 · 4. EDA 분석

- 강의 페이지: `Web/강좌/02_반도체_공정_데이터분석/반도체_공정_데이터분석.html` → 목차 **EDA 분석**
- 새 컬럼·조건 필터·groupby·히스토그램·박스플롯·히트맵으로 합격/불합격을 비교합니다.
- `반도체_공정_샘플.csv`가 이 노트북과 같은 폴더에 있어야 합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

### 1 분석에 필요한 새 컬럼 만들기
- 설명: 1 분석에 필요한 새 컬럼 만들기 단계에서 사용하는 핵심 코드를 실행해 봅니다.

### 준비 · 1~3단계 코드 실행 (df_clean 만들기)

In [ ]:
# 📦 분석에 사용할 라이브러리 불러오기
# 'as pd'는 앞으로 pandas를 pd라는 짧은 이름으로 부르겠다는 약속
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 한글 설정 (Mac은 'AppleGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False   # 음수(-) 기호 깨짐 방지

df = pd.read_csv('반도체_공정_샘플.csv')

# 3단계: 숫자 결측값은 평균으로 채우고 중복 제거
df_filled = df.copy()
numeric_cols = ['온도_섭씨', '압력_Pa', '가스유량_slm', '전력_W',
                '진공도_mTorr', '두께_nm', '습도_pct', '진동_mm_s',
                '처리시간_sec', '냉각수온도_섭씨']
for col in numeric_cols:
    df_filled[col] = df_filled[col].fillna(df_filled[col].mean())
df_clean = df_filled.drop_duplicates().reset_index(drop=True)

print(f"준비 완료: df_clean {df_clean.shape[0]}행 × {df_clean.shape[1]}열")


In [ ]:
# df_clean을 앞으로 계속 사용 (결측·중복 처리된 깨끗한 표)

# 합격여부 1/-1 → '합격'/'불합격' 문자열로 바꿔 보기 좋게
df_clean['result'] = df_clean['합격여부'].map({1: '합격', -1: '불합격'})

# 결과 확인
df_clean[['온도_섭씨', '두께_nm', '합격여부', 'result']].head()

### 2 조건에 맞는 행만 골라내기
- 설명: 조건식을 사용해 관심 있는 행만 추출합니다.

In [ ]:
# ① 불합격 데이터만 골라내기
fail_df = df_clean[df_clean['합격여부'] == -1]
print(f"불합격 건수: {len(fail_df)}건")

In [ ]:
# ② 온도가 305도를 넘는 공정
high_temp_df = df_clean[df_clean['온도_섭씨'] > 305]
print(f"고온 공정 수: {len(high_temp_df)}건")

In [ ]:
# ③ 여러 조건 동시에 (AND는 &, OR은 |, 괄호 필수!)
condition = (df_clean['온도_섭씨'] > 305) & (df_clean['두께_nm'] > 103)
risk_df = df_clean[condition]
print(f"고온+두꺼움 공정: {len(risk_df)}건")

### 3 groupby — 그룹별로 통계 내기
- 설명: 그룹별 평균과 통계를 구해 조건별 차이를 비교합니다.

In [ ]:
# 숫자 컬럼 리스트 (3단계에서 정의한 것과 동일)
numeric_cols = ['온도_섭씨', '압력_Pa', '가스유량_slm', '전력_W',
                '진공도_mTorr', '두께_nm', '습도_pct', '진동_mm_s',
                '처리시간_sec', '냉각수온도_섭씨']

# 합격여부별 모든 컬럼의 평균
group_mean = df_clean.groupby('result')[numeric_cols].mean().round(2)
print(group_mean.T)   # .T 는 행·열 뒤집기 (보기 편하게)

### 5 히스토그램 — 값이 어떻게 퍼져 있는지 보기
- 설명: 히스토그램으로 값의 분포를 시각화합니다.

In [ ]:
# 온도 분포 히스토그램
plt.figure(figsize=(8, 4))
plt.hist(df_clean['온도_섭씨'], bins=20, color='steelblue', edgecolor='white')

plt.title('온도 분포')
plt.xlabel('온도 (°C)')
plt.ylabel('빈도 (개수)')
plt.grid(True, alpha=0.3)
plt.show()

### 6 박스플롯 — 합격 vs 불합격 한눈에 비교
- 설명: 박스플롯으로 그룹별 분포와 이상치를 비교합니다.

In [ ]:
# 합격/불합격별 온도 분포를 박스플롯으로
plt.figure(figsize=(7, 5))
sns.boxplot(data=df_clean, x='result', y='온도_섭씨',
            hue='result', legend=False,
            palette=['#52B788', '#E76F51'])

plt.title('판정별 온도 분포 비교')
plt.ylabel('온도 (°C)')
plt.grid(True, alpha=0.3)
plt.show()

### 7 상관관계 히트맵 — 변수들 간의 관계
- 설명: 상관관계 행렬을 히트맵으로 그려 변수 간 관계를 확인합니다.

In [ ]:
# 숫자 컬럼만 골라 상관관계 계산
target_cols = numeric_cols + ['합격여부']
corr_matrix = df_clean[target_cols].corr()

# 히트맵 그리기
plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix,
            annot=True,         # 숫자 표시
            fmt='.2f',          # 소수점 2자리
            cmap='RdYlGn',      # 빨강-노랑-초록
            center=0,           # 0을 중앙(노란색)으로
            linewidths=0.5)
plt.title('변수 간 상관관계')
plt.show()

## 마무리

- `groupby()` 평균표에서 불합격 공정은 온도·두께·진공도·전력이 조금씩 높았습니다.
- 히트맵의 **합격여부** 행에서 절댓값이 큰 변수를 찾아보세요.
- 상관관계와 평균 차이는 **원인 확정이 아니라 추가 점검이 필요한 단서**입니다.